# Seed Capture Count

## A script to count the number of Wayback captures for each seed URL in a given collection.
### By: Cecilia Knaub, Product Support Specliaist, Archive-It

In [ ]:
# Import the requests library, which is used to make the API Call
import requests

In [ ]:
# Import the session file. This manages request rates to ensure friendly access.
!python throttled_session.py
from throttled_session import ThrottledSession

## User input: Modify these variables before running the script

<ul>
    <li> <strong>Collection ID (collection_id):</strong> Found in the public collection url (e.g., https://archive-it.org/collections/6808 -> 6808). <em>Enter as an integer.</em></li>
    <li>
        <strong>File path to Seed Url file (seed_url_file_path):</strong> The path where your bulk exported seed url file is saved. The file must be in .csv format with a header row. <em>Enter as a string, using f string notation (e.g., f"filepath")</em>
    </li>
    <li>
        <strong>Output File Name (output_file_name):</strong> What you want to call the output file. This doesn't has to be changed. <em>Enter as a string, using f string notation (e.g., f"filepath")</em>
    </li>
    <li>
       <strong> Outout File Path:</strong> Where you want to save the output file.  <em>Enter as a string, using f string notation (e.g., f"filepath")</em></li>
    </li>
</ul>

In [ ]:
# User should modify the collection_id, name the output file, enter the file path to the seed metadata export (saved as csv)
# and indicate the path where they want the csv output saved.

collection_id = # Enter the collection number
seed_url_file_path = # Enter the filepath to your seed urls csv (e.g., f"/my-folder/my_urls.csv")
output_file_name = # Name the output file
output_file_path = # Swap for output file path (e.g., f"/my-folder/my_captures_collection_1.csv"

# Initialize lists to hold seed urls and count data later
seed_urls = []
captures = []

## Import the seed url file

In [ ]:
# Open the seed metadata csv file and parse the seed urls into a list.
try:
    with open(seed_url_file_path, "r") as f:
        seed_urls = [line.strip().split(',')[0] for line in f if line.strip()][1:]
    print(f"File imported ✅")
    assert f.closed

except Exception as e:
    print(f"An unexpected error occurred importing the file: {e} ❌")

File imported ✅


## Request and count the Wayback captures

In [ ]:
# Create the session
session = ThrottledSession(max_rate=2, slow_threshold=3)

# Loop through each seed URL and make a request to the Wayback CDX/C API. Count the number of results.
for seed in seed_urls:
    url = f"https://wayback.archive-it.org/{collection_id}/timemap/cdx?url={seed}&collapse=timestamp:11"
    line_count = 0

    # Indicate if the call is successful
    try:
        session.get(url)
        response = session.get(url)
        print(f"Seed URL: {seed}, Response Status Code: {response.status_code}")

        for line in response.text.splitlines():
            line_count += 1
        captures.append((seed, line_count))

    # If the call returns an error, say that there is an error and move to the next seed.
    except requests.exceptions.RequestException as e:
        print(f"Error fetching URL {seed}: {e}. Moving to the next seed.")
        continue

Seed URL: http://www.archive-it.org/, Response Status Code: 200
Seed URL: https://archive-it.org/blog/, Response Status Code: 200


## Save the capture counts to a csv file

In [ ]:
#Save the seed url and counts as a csv.
try:
    with open(f'{output_file_name}', "w") as f:
            for row in captures:
                csv_row = ",".join(str(item) for item in row)
                f.write(csv_row + "\n")
            print(f"File saved at {output_file_name} ✅")
    assert f.closed

except Exception as e:
        print(f"An unexpected error occurred saving the file: {e} ❌")

File saved at captures_2114 ✅
